In this blog post, I want to introduce a very useful tool: py3Dmol. py3Dmol is python wrapper for the 3Dmol javascript library, which can be used to create visualizations of molecular systems. I find this tool incredibly useful because it means we can visualize a system inside a jupyter notebook. This means that when you're setting up a system, or analyzing results about it, you don't have to leave the notebook, which means you get quick feedback, and can easily share results. For this reason, I've added it to the PyBigDFT library as the InlineVisualizor class.

Unfortunately, the documentation is lacking. I think the developers are mainly concerned with making it easy for you to show a molecule on a website. But when using it from a jupyter notebook, you're inevitably going to do more complex things. In this blog post, I want to show some uses of it.

Basics
The first thing you need to know about using py3Dmol is that you should always think about structure in terms of a file format. That is the most straight forward way to use its API. So let's try reading in a PDB file...

In [1]:
!pip install py3Dmol

In [2]:
import py3Dmol

In [3]:
with open("5U6P.pdb") as ifile:
    system = "".join([x for x in ifile])

Now let's try visualizing this system.

In [4]:
view = py3Dmol.view(width=400, height=300)
view.addModelsAsFrames(system)
view.setStyle({'model': -1}, {"cartoon": {'color': 'white'}})
view.zoomTo()
view.show()

Output hidden; open in https://colab.research.google.com to view.

Notice that the way parameters work when setting the style is that you pass a dictionary. This is because the javascript API itself is based on json.

#Highlighting
Now one thing I often want to do is to highlight a particular part of the system I am studying. To do that, we can manually modify the style of individual atoms.

In [5]:
view = py3Dmol.view(width=400, height=300)
view.addModelsAsFrames(system)

i = 0
for line in system.split("\n"):
    split = line.split()
    if len(split) == 0 or split[0] != "ATOM":
        continue
    if split[3] == "SER":
        color = "blue"
    else:
        color = "yellow"
    idx = int(split[1])

    view.setStyle({'model': -1, 'serial': i+1
                   }, {"cartoon": {'color': color}})
    i += 1
view.zoomTo()
view.show()

Output hidden; open in https://colab.research.google.com to view.

#Datastructures
Up until now, the code I've been presenting can be described as, at best, pretty hacky. This is because we don't have a datastructure for our molecules. So let's build one. But let's not just build any old structure, instead I want to create a constructor that gives us maximum flexibility when dealing with py3Dmol.

In my opinion, the best approach is to break down a system into two levels: molecules and atoms. We will consider molecules to be lists of atoms. And atoms will be dictionaries which can be flexible about what kind of data they store. For this blog post, we just need to write a molecule and atom that can convert to and from pdb.

In [6]:
class Atom(dict):
    def __init__(self, line):
        self["type"] = line[0:6].strip()
        self["idx"] = line[6:11].strip()
        self["name"] = line[12:16].strip()
        self["resname"] = line[17:20].strip()
        self["resid"] = int(int(line[22:26]))
        self["x"] = float(line[30:38])
        self["y"] = float(line[38:46])
        self["z"] = float(line[46:54])
        self["sym"] = line[76:78].strip()

    def __str__(self):
        line = list(" " * 80)

        line[0:6] = self["type"].ljust(6)
        line[6:11] = self["idx"].ljust(5)
        line[12:16] = self["name"].ljust(4)
        line[17:20] = self["resname"].ljust(3)
        line[22:26] = str(self["resid"]).ljust(4)
        line[30:38] = str(self["x"]).rjust(8)
        line[38:46] = str(self["y"]).rjust(8)
        line[46:54] = str(self["z"]).rjust(8)
        line[76:78] = self["sym"].rjust(2)
        return "".join(line) + "\n"

In [7]:
class Molecule(list):
    def __init__(self, file):
        for line in file:
            # Only process lines that start with "ATOM" or "HETATM"
            if line.startswith("ATOM") or line.startswith("HETATM"):
                try:
                    self.append(Atom(line))
                except ValueError as e:
                    # Print a warning if a line cannot be parsed as an Atom
                    print(f"Warning: Could not parse line as Atom: {line.strip()} - {e}")


    def __str__(self):
        outstr = ""
        for at in self:
            outstr += str(at)

        return outstr

In [17]:
with open("2ZJN.pdb") as ifile:
    mol = Molecule(ifile)

In [18]:
view = py3Dmol.view(width=400, height=300)
view.addModelsAsFrames(str(mol))
view.setStyle({'model': -1}, {"cartoon": {'color': 'spectrum'}})
view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Now let's get to the real meat: how do we ensure flexibility? The answer lies in the dictionary representation of the atoms. What I suggest is that we add arbitrary pymol commands at an atomic level:

In [19]:
for at in mol:
    if at["resname"] == "PRO":
        at["pymol"] = {"cartoon": {'color': "red"}}
    if at["resname"] == "GLU":
        at["pymol"] = {"cartoon": {'color': "orange"}}
    if at["resname"] == "SER":
        at["pymol"] = {"cartoon": {'color': "pink"}}
    elif at["resname"] == "ARG":
        at["pymol"] = {"cartoon": {'color': 'blue'}}

Then we can access that when we are performing our visualization.

In [20]:
view = py3Dmol.view(width=400, height=300)
view.addModelsAsFrames(str(mol))
for i, at in enumerate(mol):
    default = {"cartoon": {'color': 'white'}}
    view.setStyle({'model': -1, 'serial': i+1}, at.get("pymol", default))

view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.